# D8 (PacBio family trio) — 11-tool proof-of-concept accuracy

RNA-seq FASTQs for the mother/father/daughter trio were STAR-aligned and typed
with 11 of the 12 pipeline tools (arcasHLA, OptiType, T1K, seq2HLA, RNA2HLA,
HLAminer, HLAforest, HLA-VBSeq, HISAT-genotype, HLApers, HLA-HD). PHLAT is excluded: its Python
code was recovered from a third-party Docker image (spastors/phlat_env) after
the official site and known mirrors turned out to be login-walled, but its
required ~3.5GB Bowtie2 reference index (`index4phlat`, a synthetic
pseudo-chr6 build with hardcoded coordinate offsets) was never independently
redistributed and both known Google Drive sources for it are now access-
restricted -- no working mirror found. HLA-HD was obtained after manual
registration approval from Kyoto University and added afterward.

**Caveat carried through every number below:** these FASTQs are far shallower
than the main D1-D7 cohort (197k-238k read pairs/sample vs. typically
millions), so most tools' HLA-A/HLA-C calls are no-calls purely from read
depth, not tool failure. Treat this as a pipeline proof-of-concept, not a
like-for-like accuracy comparison with the main cohort.

Scoring reuses the exact contract from `accuracy_fixed_executed.ipynb`
(`reformat_allele`, `build_global_valid`, `score_pair`) — 2-field resolution,
unfiltered (novel calls count as errors).

In [1]:
import glob
import json
import os
import re

import pandas as pd

RAW = "../datasets/raw"
SAMPLES = ["mother", "father", "daughter"]
GS_LOCI = ["A", "B", "C", "DQB1", "DRB1"]
GS_PATH = "../datasets/8_gs.csv"
PR_DIR = "../results/standard"

## 1) Canonical scoring helpers (verbatim from `accuracy_fixed_executed.ipynb`)

In [2]:
def reformat_allele(allele: str, resolution: int = 2):
    a = allele.strip()
    if "*" not in a:
        return a
    gene, rest = a.split("*", 1)
    parts = rest.split(":")
    if resolution == 1:
        return f"{gene}*{parts[0]}"
    else:
        if len(parts) < 2:
            return f"{gene}*{parts[0]}"
        if parts[1].lower() == "xx":
            return None
        return f"{gene}*{parts[0]}:{parts[1]}"


def get_dynamic_loci(gs_df: pd.DataFrame) -> list:
    return sorted({c.split(".")[0] for c in gs_df.columns[1:]})


def build_global_valid(gs_df: pd.DataFrame, resolution: int = 2) -> dict:
    loci = get_dynamic_loci(gs_df)
    valid = {L: set() for L in loci}
    for _, row in gs_df.iterrows():
        for L in loci:
            for suf in ("", ".1"):
                col = f"{L}{suf}"
                if col in row and pd.notna(row[col]):
                    for a in str(row[col]).split("/"):
                        fa = reformat_allele(a, resolution)
                        if fa is not None:
                            valid[L].add(fa)
    return valid


def score_pair(gs_raw, pr_raw, valid_global, resolution=2):
    gs_set = {reformat_allele(a, resolution) for a in gs_raw}
    gs_set.discard(None)
    if not gs_set:
        return None
    if len(pr_raw) == 0:
        pair = [None, None]
    elif len(pr_raw) == 1:
        pair = [pr_raw[0], pr_raw[0]]
    else:
        pair = pr_raw[:2]
    correct = miscalled = novel = no_call = 0
    for allele in pair:
        if allele is None:
            no_call += 1
        else:
            fmt = reformat_allele(allele, resolution)
            if fmt is None or fmt not in valid_global:
                novel += 1
            elif fmt in gs_set:
                correct += 1
            else:
                miscalled += 1
    return correct, miscalled, novel, no_call

## 2) Gold standard (trio rows only)

In [3]:
gs_all = pd.read_csv(GS_PATH, dtype=str)
gs_all["Run"] = gs_all["Run"].str.strip()
gs_trio = gs_all[gs_all["Run"].isin(SAMPLES)].reset_index(drop=True)
gs_trio

,Run,A,A.1,B,B.1,C,C.1,DQB1,DQB1.1,DRB1,DRB1.1
0,mother,A*02:01:01:01,A*02:01:01:01,B*40:01:02:04,B*18:01:01:52,C*03:04:01:01,C*07:01:01:16,DQB1*06:02:01:01,DQB1*03:01:01:03,DRB1*15:01:01:03,DRB1*11:04:01:new
1,father,A*24:02:01:05,A*26:01:01:01,B*38:01:01:01,B*13:02:01:01,C*12:03:01:01,C*06:02:01:01,DQB1*02:02:01:01,DQB1*03:02:01:01,DRB1*07:01:01:01,DRB1*04:02:new
2,daughter,A*26:01:01:01,A*02:01:01:01,B*38:01:01:01,B*18:01:01:52,C*12:03:01:01,C*07:01:01:16,DQB1*03:01:01:03,DQB1*03:02:01:01,DRB1*11:04:01:03,DRB1*04:02:new


## 3) Per-tool output parsers

Each tool's raw output has a different format; each parser below extracts,
per sample and per GS locus, up to two allele strings in `GENE*field1:field2[:...]`
form. Where a tool reports an ambiguity group for one slot (arcasHLA/T1K/HLApers
comma- or hyphen-joined lists), the **first listed member is taken as the
representative allele** for that slot — a simplification, not the full
ambiguous-match handling the main pipeline uses for GS-side ambiguity.

In [4]:
def empty_calls():
    return {L: [] for L in GS_LOCI}


def parse_arcashla(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_arcashla/{sample}_Aligned.genotype.json"
    d = json.load(open(path))
    for L in GS_LOCI:
        if L in d:
            calls[L] = d[L][:2]
    return calls


def parse_t1k(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_t1k/{sample}_genotype.tsv"
    with open(path) as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            gene = p[0].replace("HLA-", "")
            if gene not in GS_LOCI:
                continue
            for idx in (2, 5):
                if idx < len(p) and p[idx] not in (".", ""):
                    allele = p[idx].split(",")[0].replace("HLA-", "")
                    calls[gene].append(allele)
    return calls


def parse_optitype(sample):
    calls = empty_calls()
    matches = glob.glob(f"{RAW}/d8_optitype/{sample}/*/*_result.tsv")
    if not matches:
        return calls
    with open(matches[0]) as f:
        header = f.readline().rstrip("\n").split("\t")
        row = f.readline().rstrip("\n").split("\t")
    d = dict(zip(header, row))
    for L, cols in [("A", ("A1", "A2")), ("B", ("B1", "B2")), ("C", ("C1", "C2"))]:
        for c in cols:
            if d.get(c):
                calls[L].append(d[c])
    return calls


def _parse_seq2hla_style(path):
    out = {}
    if not os.path.exists(path):
        return out
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or line.startswith("#") or line.startswith("---"):
                continue
            p = line.split("\t")
            if len(p) < 5:
                continue
            locus, a1, _, a2, _ = p[:5]
            alleles = []
            for a in (a1, a2):
                a = a.replace("\x27", "").strip()
                if a and a != "no":
                    alleles.append(a)
            if alleles:
                out[locus] = alleles
    return out


def parse_seq2hla(sample):
    calls = empty_calls()
    for suffix in ("-ClassI-class.HLAgenotype4digits", "-ClassI-nonclass.HLAgenotype4digits", "-ClassII.HLAgenotype4digits"):
        d = _parse_seq2hla_style(f"{RAW}/d8_seq2hla/{sample}{suffix}")
        for L in GS_LOCI:
            if L in d:
                calls[L] = d[L][:2]
    return calls


def parse_rna2hla(sample):
    calls = empty_calls()
    d = _parse_seq2hla_style(f"{RAW}/d8_rnaseq/d8trio/{sample}.final.HLAgenotype4digits.txt")
    for L in GS_LOCI:
        if L in d:
            calls[L] = d[L][:2]
    return calls


def parse_hlaminer(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_hlaminer/{sample}/HLAminer_HPRA.csv"
    if not os.path.exists(path):
        return calls
    current_gene = None
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            m = re.match(r"^HLA-([A-Z0-9]+)\s*$", line)
            if m:
                current_gene = m.group(1)
                continue
            m3 = re.match(r"^\s*([A-Z0-9]+\*[\d:]+P?),", line)
            if m3 and current_gene in GS_LOCI and len(calls[current_gene]) < 2:
                calls[current_gene].append(m3.group(1))
    return calls


def parse_hlaforest(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_hlaforest/{sample}/haplotypes.txt"
    if not os.path.exists(path):
        return calls
    with open(path) as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 2:
                continue
            last = fields[-1]
            parts = last.split(":")
            if len(parts) < 2 or parts[0] != "ROOT":
                continue
            gene = parts[1]
            if gene not in GS_LOCI:
                continue
            allele = f"{gene}*{':'.join(parts[2:])}"
            if len(calls[gene]) < 2:
                calls[gene].append(allele)
    return calls


def load_hlavbseq_allelelist():
    mapping = {}
    path = "../../tools/HLA-VBSeq/Allelelist_v2.txt"
    with open(path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 2:
                mapping[p[0]] = p[1]
    return mapping


_VBSEQ_MAP = load_hlavbseq_allelelist()


def parse_hlavbseq(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_hlavbseq/{sample}_result.txt"
    per_gene = {L: [] for L in GS_LOCI}
    with open(path) as f:
        next(f)
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) < 5:
                continue
            hla_id = p[0].replace("HLA:", "")
            theta = float(p[4])
            allele = _VBSEQ_MAP.get(hla_id)
            if not allele or "*" not in allele:
                continue
            gene = allele.split("*")[0]
            if gene in GS_LOCI:
                per_gene[gene].append((theta, allele))
    for L in GS_LOCI:
        top = sorted(per_gene[L], key=lambda x: -x[0])[:2]
        calls[L] = [a for _, a in top]
    return calls


def parse_hlapers(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_hlapers/{sample}/genotypes.tsv"
    if not os.path.exists(path):
        return calls
    with open(path) as f:
        next(f)
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) < 2:
                continue
            locus, allele_group = p[0], p[1]
            if locus not in GS_LOCI:
                continue
            first = allele_group.split("-")[0].replace("IMGT_", "")
            if len(calls[locus]) < 2:
                calls[locus].append(first)
    return calls


def parse_hisatgeno(sample):
    # CORRECTED 2026-09-02: originally attributed to low read depth, but
    # later investigation (during the D7 tooling pass) found this
    # installation of hisat-genotype has two real upstream bugs (a
    # str-vs-list mishandling of --base, and a missing hisat2/VERSION file
    # expected by the genotyping stage) that make it silently produce empty
    # output for ANY input, regardless of depth -- including these D8
    # samples. Even after patching both, the genotyping step itself
    # deadlocks (0% CPU for 24h+ on a test run) before producing calls. So
    # this remains a no-call here too, but the cause is a broken tool, not
    # insufficient D8 read depth.
    return empty_calls()


def parse_hlahd(sample):
    calls = empty_calls()
    path = f"{RAW}/d8_hlahd/{sample}/result/{sample}_final.result.txt"
    if not os.path.exists(path):
        return calls
    with open(path) as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) < 3:
                continue
            gene = p[0]
            if gene not in GS_LOCI:
                continue
            a1, a2 = p[1], p[2]
            if a1 == "Not typed":
                continue
            a1 = a1.replace("HLA-", "")
            calls[gene].append(a1)
            if a2 not in ("-", "Not typed"):
                calls[gene].append(a2.replace("HLA-", ""))
            elif a2 == "-":
                calls[gene].append(a1)
    return calls


PARSERS = {
    "arcas": parse_arcashla,
    "t1k": parse_t1k,
    "optitype": parse_optitype,
    "seq2hla": parse_seq2hla,
    "rna2hla": parse_rna2hla,
    "hlaminer": parse_hlaminer,
    "hlaforest": parse_hlaforest,
    "hlavbseq": parse_hlavbseq,
    "hlapers": parse_hlapers,
    "hisat": parse_hisatgeno,
    "hlahd": parse_hlahd,
}

## 4) Write predictions in the pipeline's standard `results/standard/{tool}_d8.csv` format

In [5]:
os.makedirs(PR_DIR, exist_ok=True)

pred_cols = ["Run"] + [c for L in GS_LOCI for c in (L, f"{L}.1")]

for tool, parser in PARSERS.items():
    rows = []
    for s in SAMPLES:
        calls = parser(s)
        row = {"Run": s}
        for L in GS_LOCI:
            vals = calls.get(L, [])
            row[L] = vals[0] if len(vals) > 0 else None
            row[f"{L}.1"] = vals[1] if len(vals) > 1 else None
        rows.append(row)
    df = pd.DataFrame(rows, columns=pred_cols)
    df.to_csv(f"{PR_DIR}/{tool}_d8.csv", index=False)

print("wrote", len(PARSERS), "prediction files to", PR_DIR)

wrote 11 prediction files to ../results/standard


## 5) Score each tool against the trio gold standard (2-field, unfiltered)

In [6]:
valid_global = build_global_valid(gs_trio, resolution=2)

summary_rows = []
detail_rows = []

for tool in PARSERS:
    pr_df = pd.read_csv(f"{PR_DIR}/{tool}_d8.csv", dtype=str)
    merged = pd.merge(gs_trio, pr_df, left_on="Run", right_on="Run", how="inner", suffixes=("", "_pre"))

    tot_correct = tot_slots = 0
    for _, row in merged.iterrows():
        for L in GS_LOCI:
            gs_raw = [row[c] for c in (L, f"{L}.1") if pd.notna(row.get(c))]
            if not gs_raw:
                continue
            pr_raw = [row[c] for c in (f"{L}_pre", f"{L}.1_pre") if pd.notna(row.get(c))]
            res = score_pair(gs_raw, pr_raw, valid_global[L], resolution=2)
            if res is None:
                continue
            correct, miscalled, novel, no_call = res
            tot_correct += correct
            tot_slots += 2
            detail_rows.append({
                "tool": tool, "sample": row["Run"], "locus": L,
                "correct": correct, "miscalled": miscalled,
                "novel": novel, "no_call": no_call,
            })
    summary_rows.append({
        "tool": tool,
        "correct_slots": tot_correct,
        "total_slots": tot_slots,
        "accuracy_2field_unfiltered": round(100 * tot_correct / tot_slots, 1) if tot_slots else float("nan"),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("accuracy_2field_unfiltered", ascending=False).reset_index(drop=True)
detail_df = pd.DataFrame(detail_rows)
summary_df

,tool,correct_slots,total_slots,accuracy_2field_unfiltered
0,t1k,8,30,26.7
1,rna2hla,8,30,26.7
2,seq2hla,8,30,26.7
3,hlahd,8,30,26.7
4,arcas,6,30,20.0
5,hlavbseq,6,30,20.0
6,optitype,5,30,16.7
7,hlaforest,5,30,16.7
8,hlapers,4,30,13.3
9,hlaminer,0,30,0.0


## 6) Per-tool × per-sample breakdown

In [7]:
pivot = detail_df.assign(slot_correct=detail_df["correct"]).groupby(
    ["tool", "sample"]
)[["correct", "no_call", "miscalled", "novel"]].sum()
pivot["total"] = pivot.sum(axis=1)
pivot["accuracy_pct"] = (100 * pivot["correct"] / pivot["total"]).round(1)
pivot

correct  no_call  miscalled  novel  total  accuracy_pct
tool      sample                                                           
arcas     daughter        2        6          0      2     10          20.0
          father          0        8          0      2     10           0.0
          mother          4        4          0      2     10          40.0
hisat     daughter        0       10          0      0     10           0.0
          father          0       10          0      0     10           0.0
          mother          0       10          0      0     10           0.0
hlaforest daughter        1        6          0      3     10          10.0
          father          2        4          0      4     10          20.0
          mother          2        6          0      2     10          20.0
hlahd     daughter        2        8          0      0     10          20.0
          father          0       10          0      0     10           0.0
          mother          6        4          0      0     10          60.0
hlaminer  daughter        0        8          0      2     10           0.0
          father          0        6          0      4     10           0.0
          mother          0        6          0      4     10           0.0
hlapers   daughter        1        2          0      7     10          10.0
          father          1        2          0      7     10          10.0
          mother          2        2          0      6     10          20.0
hlavbseq  daughter        0        0          2      8     10           0.0
          father          4        0          0      6     10          40.0
          mother          2        0          2      6     10          20.0
optitype  daughter        1        6          0      3     10          10.0
          father          3        4          0      3     10          30.0
          mother          1        6          0      3     10          10.0
rna2hla   daughter        2        8          0      0     10          20.0
          father          2        6          0      2     10          20.0
          mother          4        6          0      0     10          40.0
seq2hla   daughter        2        8          0      0     10          20.0
          father          2        6          0      2     10          20.0
          mother          4        4          0      2     10          40.0
t1k       daughter        2        6          0      2     10          20.0
          father          2        4          0      4     10          20.0
          mother          4        4          0      2     10          40.0

In [8]:
summary_df.to_csv(f"{PR_DIR}/../d8_trio_accuracy_summary.csv", index=False)
detail_df.to_csv(f"{PR_DIR}/../d8_trio_accuracy_detail.csv", index=False)
print("saved summary + detail CSVs to results/")

saved summary + detail CSVs to results/
